# Silver Layer

Read from Bronze, clean and validate, add features

1. Read from bronze_carbon_intensity_regional
2. Rename period_from → settlement_period, drop period_to
3. Trim whitespace on string columns
4. Validate carbon_intensity is non-negative (it should never be below 0)   
5. Flag any nulls rather than filling them
6. Write to silver_carbon_intensity_regional  

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from functools import reduce

In [0]:
# Read from bronze_carbon_intensity_regional

df_silver = spark.read.table("bronze_carbon_intensity_regional")

print(f"Rows: {df_silver.count()}")
print(f"Columns: {len(df_silver.columns)}") 


In [0]:
# Write to silver_carbon_intensity_regional
df_silver = df_silver.withColumnRenamed("period_from", "settlement_period").drop("period_to")

# Define key cols before adding more columns
key_cols = [field.name for field in df_silver.schema.fields]


In [0]:
# Step 3: trim whitespace on string columns
string_cols = [field.name for field in df_silver.schema.fields if isinstance(field.dataType, StringType)]

for str_col in string_cols:
    df_silver = df_silver.withColumn(str_col, F.trim(F.col(str_col)))


In [0]:
# step 4 Validate carbon_intensity is non-negative (it should never be below 0) 

df_silver = df_silver.withColumn("carbon_intensity_valid", F.col("carbon_intensity") >= 0)

In [0]:
# step 5 Flag any nulls rather than filling them

has_nulls = [F.col(col_name).isNull()  for col_name in key_cols]
reduce(lambda a, b: a | b, has_nulls)
df_silver = df_silver.withColumn("contains_nulls", reduce(lambda a, b: a | b, has_nulls))

In [0]:
df_silver.head(10)


In [0]:
display(df_silver.describe().limit(25))

In [0]:
# Step 6 Save as a delt table 

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_carbon_intensity_regional")